# HStream Extractor (Colab)

Bulk downloader + subtitle remuxer for hstream.moe

Fill in **XSRF-TOKEN** and **hstream_session** from your browser cookies (DevTools → Application → Cookies → hstream.moe).
Do not share notebooks that contain your real cookies.

In [ ]:
import os
import subprocess
import requests
import glob
from tqdm.notebook import tqdm

print("Installing dependencies...")
try:
    subprocess.run(["pip", "install", "--upgrade", "yt-dlp", "requests", "tqdm"], check=True)
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "aria2", "ffmpeg"], check=True)
    print("Dependencies installed successfully!")
except Exception as e:
    print(f"Warning during dependency setup: {e}")

In [ ]:
# @title Settings
URL_LIST = "https://hstream.moe/hentai/sweet-home-h-na-oneesan-wa-suki-desu-ka-1 https://hstream.moe/hentai/sweet-home-h-na-oneesan-wa-suki-desu-ka-2 https://hstream.moe/hentai/sweet-home-h-na-oneesan-wa-suki-desu-ka-3"  #@param {type:"string"}
DESTINATION_FOLDER = "/content/downloads"  #@param {type:"string"}
XSRF_TOKEN = ""  #@param {type:"string"}
HSTREAM_SESSION = ""  #@param {type:"string"}

print("Settings loaded")
print("URLs:", URL_LIST)
print("Destination:", DESTINATION_FOLDER)
print("XSRF-TOKEN set:", bool(XSRF_TOKEN.strip()))
print("hstream_session set:", bool(HSTREAM_SESSION.strip()))

In [ ]:
if not os.path.exists(DESTINATION_FOLDER):
    os.makedirs(DESTINATION_FOLDER)

urls = [u.strip() for u in URL_LIST.replace("\n", " ").split() if u.strip()]
print(f"Found {len(urls)} links to process.\n")

# Build Cookie header from form fields (same idea as original COOKIE_HEADER)
cookie_parts = []
if XSRF_TOKEN.strip():
    cookie_parts.append(f"XSRF-TOKEN={XSRF_TOKEN.strip()}")
if HSTREAM_SESSION.strip():
    cookie_parts.append(f"hstream_session={HSTREAM_SESSION.strip()}")
COOKIE_HEADER = "; ".join(cookie_parts)

if not COOKIE_HEADER:
    print("WARNING: No cookies set. Many hstream.moe links will fail without login cookies.")

for index, url in enumerate(tqdm(urls, desc="Overall Progress", unit="video"), start=1):
    tqdm.write(f"\nProcessing [{index}/{len(urls)}]: {url}")

    output_template = os.path.join(DESTINATION_FOLDER, "%(title)s.%(ext)s")

    cmd = [
        "yt-dlp", "-v", "--downloader", "aria2c",
        "--concurrent-fragments", "8",
        "-o", output_template,
    ]
    if COOKIE_HEADER:
        cmd += ["--add-header", f"Cookie: {COOKIE_HEADER}"]
    cmd.append(url)

    try:
        subprocess.run(cmd, check=True)
    except subprocess.CalledProcessError as e:
        tqdm.write(f"Error downloading video for {url}: {e}")
        continue

    files = glob.glob(os.path.join(DESTINATION_FOLDER, "*"))
    if not files:
        tqdm.write("No output files found. Skipping...")
        continue

    latest_video = max(files, key=os.path.getctime)
    base_name = os.path.splitext(os.path.basename(latest_video))[0]
    final_mkv = os.path.join(DESTINATION_FOLDER, f"{base_name}.mkv")

    if latest_video == final_mkv:
        continue

    # Derive episode number and series slug from URL (same style as original)
    parts = url.rstrip("/").split("/")
    ep_num = parts[-1].split("-")[-1]
    series_name_clean = "-".join(parts[-1].split("-")[:-1]) if "-" in parts[-1] else parts[-1]
    # External host often uses dots; try both styles if needed
    series_dot = series_name_clean.replace("-", ".")

    sub_path = os.path.join(DESTINATION_FOLDER, f"{base_name}.ass")
    tqdm.write("Downloading subtitle...")

    sub_ok = False
    for slug in (series_dot, series_name_clean):
        sub_url = f"https://oppai-str.shoujo-h.org/2024/{slug}/E{int(ep_num):02d}/eng.ass"
        try:
            sub_res = requests.get(sub_url, stream=True, timeout=30)
            if sub_res.status_code == 200:
                total_size = int(sub_res.headers.get("content-length", 0))
                with open(sub_path, "wb") as f, tqdm(
                    desc="Subtitle Progress",
                    total=total_size,
                    unit="B",
                    unit_scale=True,
                    unit_divisor=1024,
                    leave=False,
                ) as sub_bar:
                    for data in sub_res.iter_content(1024):
                        sub_bar.update(len(data))
                        f.write(data)
                sub_ok = True
                break
        except Exception:
            pass

    if sub_ok:
        try:
            tqdm.write("Remuxing video and subtitles into MKV...")
            subprocess.run([
                "ffmpeg", "-y", "-i", latest_video, "-i", sub_path,
                "-map", "0", "-map", "1", "-c", "copy",
                "-metadata:s:s:0", "language=eng", final_mkv
            ], check=True)

            if os.path.exists(sub_path):
                os.remove(sub_path)
            if latest_video != final_mkv and os.path.exists(latest_video):
                os.remove(latest_video)

            tqdm.write(f"Successfully Saved: {final_mkv}")
        except Exception as ex:
            tqdm.write(f"Error processing subtitle/remux: {ex}")
    else:
        tqdm.write("Subtitle not found. Keeping original video.")

print("\n" + "=" * 50)
print("ALL TASKS COMPLETED!")

In [ ]:
!zip -r /content/hstream_downloads.zip {DESTINATION_FOLDER}
print("Created: /content/hstream_downloads.zip")
print("Download from left sidebar -> Files")